# Inference Demo

This notebook loads a PA image from `CT_RATE_demo_data` and runs CheXanatomy inference with either a local adapter path or a direct model id.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from peft import PeftConfig, PeftModel
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'paligemma_training_data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

UTILS_DIR = PROJECT_ROOT / 'utils'
if str(UTILS_DIR) not in sys.path:
    sys.path.append(str(UTILS_DIR))

DEMO_DATA_ROOT = PROJECT_ROOT / 'CT_RATE_demo_data'
MODEL_PATH_ENV = os.environ.get('CHEXANATOMY_MODEL_PATH')
MODEL_PATH_OR_ID = Path(MODEL_PATH_ENV) if MODEL_PATH_ENV else 'google/paligemma2-3b-pt-224'
PROJECTION = 'PA'
SAMPLE_SUBDIR = 'train_1_a_2'
PROMPT = 'segment left lung'

print(f'Project root: {PROJECT_ROOT}')
print(f'Demo root: {DEMO_DATA_ROOT}')
print(f'Model path or id: {MODEL_PATH_OR_ID}')
print('Set CHEXANATOMY_MODEL_PATH to a fine-tuned adapter path for segmentation-quality results.')

In [ ]:
def resolve_model(model_path_or_id):
    if isinstance(model_path_or_id, Path) and model_path_or_id.exists():
        peft_config = PeftConfig.from_pretrained(str(model_path_or_id))
        base_model_id = peft_config.base_model_name_or_path
        base_model = PaliGemmaForConditionalGeneration.from_pretrained(base_model_id)
        model = PeftModel.from_pretrained(base_model, str(model_path_or_id))
        processor = PaliGemmaProcessor.from_pretrained(base_model_id)
        return model, processor, base_model_id

    base_model_id = str(model_path_or_id)
    model = PaliGemmaForConditionalGeneration.from_pretrained(base_model_id)
    processor = PaliGemmaProcessor.from_pretrained(base_model_id)
    return model, processor, base_model_id

model, processor, base_model_id = resolve_model(MODEL_PATH_OR_ID)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()
print('Using device:', device)
print('Base model:', base_model_id)

In [ ]:
image_path = DEMO_DATA_ROOT / f'CT_RATE_projections_{PROJECTION}' / SAMPLE_SUBDIR / 'ct.png'
image = Image.open(image_path).convert('RGB')

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title(f'{PROJECTION} image: {image_path.name}')
plt.axis('off')
plt.show()

In [ ]:
full_prompt = PROMPT.strip()
inputs = processor(image, full_prompt, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

raw_response = processor.decode(outputs[0], skip_special_tokens=False)
generated_text = processor.decode(outputs[0], skip_special_tokens=True)
answer = generated_text[len(full_prompt):].strip() if generated_text.startswith(full_prompt) else generated_text

print('Prompt:', full_prompt)
print('Generated text:', generated_text[:500])
print('Answer:', answer[:500])

## Visualize the reconstructed segmentation mask

The next cell decodes the segmentation tokens in the model response, reconstructs the mask, and compares it with the ground-truth mask from the demo data.

In [ ]:
import re

os.environ['JAX_PLATFORM_NAME'] = 'cpu'

from VQVAE_decoder_utils import extract_objs


SEGMENT_PATTERN = re.compile(
    r'<loc\d{4}><loc\d{4}><loc\d{4}><loc\d{4}>\s*(?:<seg\d{3}>){16}(?:\s*[^;<>]+)?'
 )


def find_segmentation_span(text):
    if not text:
        return None
    match = SEGMENT_PATTERN.search(text)
    return match.group(0) if match else None


def prompt_to_structure_key(prompt):
    structure_text = prompt.strip().lower()
    if structure_text.startswith('segment '):
        structure_text = structure_text[len('segment '):]

    aliases = {
        'left lung': 'lung_left',
        'right lung': 'lung_right',
        'left upper lobe': 'lung_upper_lobe_left',
        'left lower lobe': 'lung_lower_lobe_left',
        'right upper lobe': 'lung_upper_lobe_right',
        'right middle lobe': 'lung_middle_lobe_right',
        'right lower lobe': 'lung_lower_lobe_right',
    }
    return aliases.get(structure_text, structure_text.replace(' ', '_'))


def load_ground_truth_mask(image_file_path, prompt):
    structure_key = prompt_to_structure_key(prompt)
    mask_path = image_file_path.parent / f'{structure_key}.png'
    if not mask_path.exists():
        return None, structure_key, mask_path

    mask_image = Image.open(mask_path).convert('L')
    mask_array = np.asarray(mask_image)
    return mask_array, structure_key, mask_path


def extract_first_mask_from_text(text, pil_image):
    span = find_segmentation_span(text)
    if span is None:
        return None, None, None

    width, height = pil_image.size
    objects = extract_objs(span, width, height)
    for obj in objects:
        mask = obj.get('mask')
        if mask is not None:
            return np.asarray(mask), obj, span
    return None, None, span


def overlay_mask_on_image(pil_image, mask, color=(255, 0, 255), alpha=0.4, threshold=0.3):
    rgb_image = np.asarray(pil_image.convert('RGB')).astype(np.float32)
    mask_binary = np.asarray(mask) > threshold

    color_array = np.zeros_like(rgb_image)
    color_array[:, :, 0] = color[0]
    color_array[:, :, 1] = color[1]
    color_array[:, :, 2] = color[2]

    blended = rgb_image.copy()
    for channel in range(3):
        blended[:, :, channel] = np.where(
            mask_binary,
            (1 - alpha) * rgb_image[:, :, channel] + alpha * color_array[:, :, channel],
            rgb_image[:, :, channel],
        )

    return blended.clip(0, 255).astype(np.uint8), mask_binary


ground_truth_mask, structure_key, ground_truth_mask_path = load_ground_truth_mask(image_path, PROMPT)
ground_truth_binary = None if ground_truth_mask is None else np.asarray(ground_truth_mask) > 0

candidate_texts = [
    ('generated_text', generated_text),
    ('raw_response', raw_response),
    ('answer', answer),
]

mask = None
decoded_object = None
matched_span = None
matched_source = None

for source_name, candidate_text in candidate_texts:
    mask, decoded_object, matched_span = extract_first_mask_from_text(candidate_text, image)
    if mask is not None:
        matched_source = source_name
        break

binary_mask = None
overlay_image = np.asarray(image.convert('RGB'))
decoded_name = 'no predicted mask'

if mask is not None:
    overlay_image, binary_mask = overlay_mask_on_image(image, mask)
    decoded_name = decoded_object.get('name') or 'unknown structure'
else:
    print('No segmentation mask could be reconstructed from the model response.')
    print('This usually means the model did not emit <loc...><seg...> tokens for this prompt/image.')
    print()
    for source_name, candidate_text in candidate_texts:
        print(f'--- {source_name} ---')
        print(candidate_text[:500] if candidate_text else '<empty>')
        print()

if ground_truth_binary is not None:
    ground_truth_overlay, _ = overlay_mask_on_image(image, ground_truth_binary.astype(np.float32), color=(0, 255, 0))
else:
    ground_truth_overlay = np.asarray(image.convert('RGB'))

comparison_overlay = np.zeros((*np.asarray(image).shape[:2], 3), dtype=np.uint8)
if ground_truth_binary is not None:
    comparison_overlay[ground_truth_binary] = (0, 255, 0)
if binary_mask is not None:
    comparison_overlay[binary_mask] = np.where(
        comparison_overlay[binary_mask].any(axis=-1, keepdims=True),
        np.array([255, 255, 0], dtype=np.uint8),
        np.array([255, 0, 255], dtype=np.uint8),
    )

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original image')
axes[0].axis('off')

axes[1].imshow(ground_truth_overlay)
axes[1].set_title(f'Ground truth ({structure_key})')
axes[1].axis('off')

axes[2].imshow(overlay_image)
axes[2].set_title(f'Prediction ({decoded_name})')
axes[2].axis('off')

axes[3].imshow(comparison_overlay)
axes[3].set_title('Overlap: green=GT, magenta=pred, yellow=both')
axes[3].axis('off')

plt.tight_layout()
plt.show()

print(f'Ground-truth mask path: {ground_truth_mask_path}')
if ground_truth_binary is None:
    print('Ground-truth mask was not found for this prompt.')
else:
    print(f'Ground-truth mask area: {int(ground_truth_binary.sum())} pixels')

if binary_mask is not None:
    print(f'Matched source: {matched_source}')
    print(f'Matched token span: {matched_span[:200]}')
    print(f'Reconstructed mask area: {int(binary_mask.sum())} pixels')
    print(f'Decoded structure label: {decoded_name}')

if ground_truth_binary is not None and binary_mask is not None:
    intersection = np.logical_and(ground_truth_binary, binary_mask).sum()
    union = np.logical_or(ground_truth_binary, binary_mask).sum()
    iou = float(intersection / union) if union else 0.0
    print(f'IoU: {iou:.4f}')